# Заказы в RetailCRM

## Окружение

In [40]:
import importlib
import json
import logging
import os
import time
from datetime import datetime
from typing import List, Optional

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator, model_validator, ConfigDict
from requests.exceptions import RequestException, Timeout

In [42]:
# 1. Важно для Jupyter: сбросить настройки логгера, если они уже были заданы
importlib.reload(logging)

# 2. Настройка логирования
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        # Сохранение в файл (файл появится в той же папке, где лежит .ipynb)
        logging.FileHandler("retail_upload.log", mode="a", encoding="utf-8"),
        # Вывод в консоль Jupyter (под ячейкой)
        logging.StreamHandler(),
    ],
)

logger = logging.getLogger(__name__)

logger.info("Логирование настроено. Записи дублируются в файл retail_upload.log")

2026-04-13 21:57:03,308 - INFO - Логирование настроено. Записи дублируются в файл retail_upload.log


In [83]:
load_dotenv(override=True)  # Загружаем переменные окружения из .env файла
SUBDOMAIN = os.getenv("SUBDOMAIN")
API_KEY = os.getenv("API_KEY")
SITE = os.getenv("SITE")
if not SUBDOMAIN or not API_KEY or not SITE:
    logger.error("Одна или несколько переменных окружения не найдены. Проверьте .env файл.")


## Загрузка сырого списка заказов

In [45]:
with open('mock_orders.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)
print(type(raw_data))
print(len(raw_data))

<class 'list'>
50


## Анализ полей входного JSON

In [42]:
from collections import Counter

def analyze_json_keys(data):
    """
    Анализирует ключи в списке словарей. 
    Показывает все ключи и частоту их появления.
    """
    if not isinstance(data, list):
        print("Ошибка: Данные должны быть списком словарей.")
        return

    total_items = len(data)
    key_counts = Counter()

    # Функция для извлечения вложенных ключей
    def get_flatten_keys(d, prefix=""):
        keys = []
        for k, v in d.items():
            full_key = f"{prefix}{k}"
            keys.append(full_key)
            if isinstance(v, dict):
                keys.extend(get_flatten_keys(v, prefix=f"{full_key}."))
            elif isinstance(v, list) and len(v) > 0 and isinstance(v[0], dict):
                # Для списков (как items) проверяем ключи первого элемента
                keys.extend(get_flatten_keys(v[0], prefix=f"{full_key}[]."))
        return keys

    # Собираем ключи со всех объектов
    for item in data:
        unique_keys = set(get_flatten_keys(item))
        key_counts.update(unique_keys)

    print(f"--- Анализ структуры ({total_items} объектов) ---")
    print(f"{'Ключ':<40} | {'Частота':<10} | {'Наличие'}")
    print("-" * 65)

    # Сортируем по популярности (сначала те, что есть у всех)
    for key, count in key_counts.most_common():
        percentage = (count / total_items) * 100
        presence = "✅ У всех" if count == total_items else f"⚠️ У {count} из {total_items}"
        print(f"{key:<40} | {count:<10} | {presence} ({percentage:.0f}%)")



In [20]:
# --- ЗАПУСК ---
# Передай сюда свой raw_data
analyze_json_keys(raw_data)

--- Анализ структуры (50 объектов) ---
Ключ                                     | Частота    | Наличие
-----------------------------------------------------------------
items                                    | 50         | ✅ У всех (100%)
items[].initialPrice                     | 50         | ✅ У всех (100%)
email                                    | 50         | ✅ У всех (100%)
delivery.address.city                    | 50         | ✅ У всех (100%)
phone                                    | 50         | ✅ У всех (100%)
orderMethod                              | 50         | ✅ У всех (100%)
customFields.utm_source                  | 50         | ✅ У всех (100%)
items[].quantity                         | 50         | ✅ У всех (100%)
delivery                                 | 50         | ✅ У всех (100%)
items[].productName                      | 50         | ✅ У всех (100%)
lastName                                 | 50         | ✅ У всех (100%)
orderType                              

## Валидация заказов для RetaiCRM и Supabase

In [ ]:
# 1. Схема торгового предложения (Offer)
class OfferModel(BaseModel):
    id: Optional[int] = None
    externalId: Optional[str] = None
    xmlId: Optional[str] = None


# 2. Схема товара в заказе (Item)
class ItemModel(BaseModel):
    productName: Optional[str] = None
    initialPrice: float = Field(default=0.0, ge=0)  # Цена не может быть отрицательной
    quantity: float = Field(default=1.0, gt=0)
    offer: Optional[OfferModel] = None

    # Валидация: либо должен быть оффер, либо название товара
    @model_validator(mode="after")
    def check_item_id(self):
        """Функция для проверки наличия идентификатора товара (либо productName, либо один из ID в offer).
        Вход: self - экземпляр модели после валидации полей
        Выход: self - возвращаем экземпляр модели, если валидация прошла успешно, иначе выбрасываем исключение с сообщением об ошибке
        """
        offer_data = self.offer.model_dump(exclude_none=True) if self.offer else {}
        if not self.productName and not offer_data:
            raise ValueError(
                "У товара должно быть либо productName, либо один из ID в offer"
            )
        return self


class DeliveryAddressModel(BaseModel):
    city: str
    text: str


class DeliveryModel(BaseModel):
    address: DeliveryAddressModel


# --- ОБЩИЕ ПОЛЯ (База) ---
class OrderBase(BaseModel):
    model_config = ConfigDict(extra="ignore")  # Игнорируем лишнее
    firstName: Optional[str] = None
    lastName: Optional[str] = None
    phone: Optional[str] = None
    email: Optional[str] = None
    number: Optional[str] = None


# --- 1. МОДЕЛЬ ДЛЯ ЗАГРУЗКИ В CRM (Upload) ---
class OrderCreate(OrderBase):
    items: List[ItemModel] = Field(..., min_length=1)
    externalId: Optional[str] = None
    orderType: str = Field(default="main")
    delivery: Optional[DeliveryModel] = None
    createdAt: Optional[datetime] = None

    # Генерация externalId, если его нет, на основе телефона и имени (для удобства текущего использования)
    # В реальной задаче нужно генерировать не повторяющиейся ID
    @model_validator(mode="before")
    @classmethod
    def generate_external_id(cls, data):
        """Если externalId не указан, генерируем его на основе телефона и имени.
        Вход:
            data - словарь данных (до валидации),
            cls - класс модели (не используется, но нужен для декоратора)
        Выход: словарь данных с гарантированным externalId"""
        if not data.get("externalId"):
            # Берем телефон (только цифры)
            phone = str(data.get("phone", "")).replace("+", "").replace(" ", "")
            # Берем имя (в нижнем регистре)
            name = str(data.get("firstName", "noname")).lower()
            # Собираем ID: телефон + имя
            # Теперь этот ID будет ОДИНАКОВЫМ при каждом запуске для этого клиента
            data["externalId"] = f"id_{phone}_{name}"
        return data

    @field_validator("orderType")
    @classmethod
    def force_main_type(cls, _):
        """Функция для поля orderType, которая всегда возвращает "main", игнорируя входящее значение.
        Вход: значение, переданное в поле orderType (может быть любым, даже None)
        Выход: строка "main" независимо от входного значения"""
        return "main"  # Что бы ни пришло в JSON, мы меняем это на "main"

    # Валидация: наличие хотя бы одного идентификатора
    @model_validator(mode="after")
    def check_ids(self):
        """Функция для проверки наличия хотя бы одного идентификатора (externalId или number).
        Вход: self - экземпляр модели после валидации полей
        Выход: self - возвращаем экземпляр модели, если валидация прошла успешно, иначе выбрасываем исключение с сообщением об ошибке
        """
        if not self.externalId and not self.number:
            raise ValueError(
                "В заказе должен быть указан хотя бы externalId или number"
            )
        return self

    # Автоматическое форматирование даты в строку для RetailCRM
    def to_retailcrm_dict(self):
        """Функция для преобразования модели в словарь, готовый для отправки в RetailCRM.
        Вход: self - экземпляр модели OrderCreate
        Выход: словарь с данными, где поле createdAt отформатировано в строку "YYYY-MM-DD HH:MM:SS" (если createdAt указано)
        """
        data = self.model_dump(exclude_none=True)
        if self.createdAt:
            data["createdAt"] = self.createdAt.strftime("%Y-%m-%d %H:%M:%S")
        return data


# --- 2. МОДЕЛЬ ДЛЯ СИНХРОНИЗАЦИИ В SUPABASE (Read) ---
# Используем её, когда скачали данные из CRM
class OrderRead(OrderBase):
    id: int  # Обязательно, так как CRM его уже выдала
    externalId: str
    status: str
    createdAt: datetime
    totalSumm: float = Field(0.0, alias="totalSumm")

    def to_supabase(self):
        """Готовим строку для SQL"""
        return {
            "id": self.id,
            "external_id": self.externalId,
            "first_name": self.firstName,
            "last_name": self.lastName,
            "phone": self.phone,
            "number": self.number,
            # "email": self.email,
            "total_sum": self.totalSumm,
            "status": self.status,
            "created_at": self.createdAt.isoformat(),
            # model_dump() здесь вернет ВООБЩЕ ВСЕ поля,
            # включая те, что мы не описали (благодаря extra='allow')
            "raw_data": self.model_dump(mode="json"),
        }


# --- Функция проверки (для создания заказа в RetailCRM) ---
def validate_with_pydantic(data_list):
    """Функция для валидации списка заказов с помощью Pydantic.
    Вход: data_list - список словарей, каждый из которых представляет заказ (как из CRM)
    Выход: кортеж из двух элементов:
        - valid_orders: список словарей, которые прошли валидацию и готовы для отправки в RetailCRM (с отформатированными датами)
        - errors: список строк с описанием ошибок для заказов, которые не прошли валидацию
    """
    valid_orders = []
    errors = []

    for index, item in enumerate(data_list):
        try:
            # Пытаемся создать модель (здесь происходит валидация)
            order = OrderCreate(**item)
            # Если успешно, сохраняем очищенный словарь
            valid_orders.append(order.to_retailcrm_dict())
        except Exception as e:
            errors.append(f"Заказ индекс {index}: {str(e)}")

    return valid_orders, errors

### Проверка на заказе с ошибкой

In [47]:
# Твой исходный JSON
test_data = [
    {
        "externalId": "101",
        "createdAt": "2024-01-01 12:00:00",
        "items": [
            {"productName": "Кроссовки", "initialPrice": 5000}
        ]
    },
    {
        "firstName": "Ошибка", 
        "items": [] # Ошибка: пустые товары и нет ID
    }
]

clean_orders, validation_errors = validate_with_pydantic(test_data)

if validation_errors:
    print(f"❌ Найдено ошибок: {len(validation_errors)}")
    for err in validation_errors:
        print(err)
else:
    print(f"✅ Все заказы ({len(clean_orders)}) валидны и готовы к отправке!")

❌ Найдено ошибок: 1
Заказ индекс 1: 1 validation error for OrderModel
items
  List should have at least 1 item after validation, not 0 [type=too_short, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.12/v/too_short


## Функция загрузки заказов на сайт

In [ ]:
def upload_orders_to_retailcrm(subdomain, api_key, orders_list, site_code):
    """
    Загрузка заказов по правилам RetailCRM v5.
    :param subdomain: только имя вашего поддомена (например, 'shop-123')
    :param api_key: ваш API-ключ для доступа к CRM
    :param orders_list: список заказов в виде словарей, готовых для отправки (после валидации и форматирования)
    :param site_code: код сайта в CRM, на который нужно загрузить заказы
    Return: список заказов в RetailCRM, которые были успешно загружены (ответ от API) или пустой список, если были ошибки при загрузке
    """

    # Формируем базовый URL строго по документации v5
    base_url = f"https://{subdomain}.retailcrm.ru/api/v5"
    endpoint = f"{base_url}/orders/upload"

    batch_size = 50
    results = []

    with requests.Session() as session:
        for i in range(0, len(orders_list), batch_size):
            batch = orders_list[i : i + batch_size]
            current_batch = i // batch_size + 1

            # Подготовка данных
            payload = {
                "apiKey": api_key,
                "site": site_code,
                "orders": json.dumps(batch),
            }

            try:
                # Отправляем запрос
                response = session.post(endpoint, data=payload, timeout=10)

                # Читаем JSON даже если статус ошибки
                try:
                    res_data = response.json()
                except:
                    res_data = {}

                # Если случилась ошибка (460, 400 и т.д.), логируем её с деталями
                if response.status_code != 200:
                    # Вытаскиваем конкретные сообщения из ответа RetailCRM
                    api_errors = res_data.get("errors", [])
                    error_msg = res_data.get("errorMsg", "Неизвестная ошибка при получении заказов из RetailCRM")

                    if api_errors:
                        # Если есть список ошибок, объединяем его в строку
                        detailed_error = "; ".join(api_errors)
                        logger.error(
                            f"Пакет {current_batch}: ОШИБКА API: {detailed_error}"
                        )
                    else:
                        logger.error(
                            f"Пакет {current_batch}: {error_msg} (Status {response.status_code})"
                        )

                response.raise_for_status()

                data = response.json()
                if data.get("success"):
                    logger.info(f"Пакет {current_batch}: Успешно загружен.")
                    results.append(data)
                else:
                    logger.error(
                        f"Пакет {current_batch}: Ошибка API: {data.get('errors')}"
                    )

            except Timeout:
                logger.warning(f"Пакет {current_batch}: Таймаут 10 сек.")
            except RequestException as e:
                logger.error(f"Пакет {current_batch}: Ошибка сети: {e}")

            time.sleep(0.2)  # Пауза между батчами
    # Вытаскиваем results даже если были ошибки, чтобы видеть, что загрузилось, а что нет
    return results

### Тестирование кода на работоспособность

In [48]:
# Чтобы дата соответствовала формату Y-m-d H:i:s из документации:
now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

my_orders = [{
    "externalId": "test_id_001",
    "firstName": "Иван",
    "lastName": "Тестовый",
    "phone": "+79991112233",
    "email": "test-order@example.com",
    "orderType": "main",
    "orderMethod": "shopping-cart",
    "status": "new",
    "items": [
      {
        "productName": "Тестовый товар #1",
        "quantity": 1,
        "initialPrice": 990
      }
    ],
    "delivery": {
      "address": {
        "city": "Москва",
        "text": "ул. Тестовая, д. 10, кв. 5"
      }
    },
    "customFields": {
      "utm_source": "jupyter_test"
    }
}]
clean_orders, validation_errors = validate_with_pydantic(my_orders)
if validation_errors:
    print(f"❌ Найдено ошибок: {len(validation_errors)}")
    for err in validation_errors:
        print(err)
else:
    print(f"✅ Все заказы ({len(clean_orders)}) валидны и готовы к отправке!")
upload_orders_to_retailcrm(SUBDOMAIN, API_KEY, clean_orders, SITE)

✅ Все заказы (1) валидны и готовы к отправке!


2026-04-13 00:46:04,322 - ERROR - Пакет 1: Ошибка сети: 460 Client Error: unknown status for url: https://noscoks.retailcrm.ru/api/v5/orders/upload


[]

## Загрузка заказов на сайт

In [ ]:
clean_orders, validation_errors = validate_with_pydantic(raw_data)

if validation_errors:
    print(f"❌ Найдено ошибок: {len(validation_errors)}")
    for err in validation_errors:
        print(err)
else:
    print(f"✅ Все заказы ({len(clean_orders)}) валидны и готовы к отправке!")

    
upload_orders_to_retailcrm(SUBDOMAIN, API_KEY, clean_orders, SITE)

2026-04-13 00:56:06,482 - ERROR - ОШИБКА СЕРВЕРА 201: {"success":true,"uploadedOrders":[{"id":45,"externalId":"id_77001234501_айгуль"},{"id":46,"externalId":"id_77012345602_дина"},{"id":47,"externalId":"id_77023456703_нургуль"},{"id":48,"externalId":"id_77034567804_малика"},{"id":49,"externalId":"id_77045678905_зарина"},{"id":50,"externalId":"id_77056789006_ирина"},{"id":51,"externalId":"id_77067890107_светлана"},{"id":52,"externalId":"id_77078901208_гульнара"},{"id":53,"externalId":"id_77089012309_анара"},{"id":54,"externalId":"id_77090123410_камила"},{"id":55,"externalId":"id_77001234511_жанар"},{"id":56,"externalId":"id_77012345612_самал"},{"id":57,"externalId":"id_77023456713_назгуль"},{"id":58,"externalId":"id_77034567814_татьяна"},{"id":59,"externalId":"id_77045678915_алия"},{"id":60,"externalId":"id_77056789016_венера"},{"id":61,"externalId":"id_77067890117_меруерт"},{"id":62,"externalId":"id_77078901218_ботагоз"},{"id":63,"externalId":"id_77089012319_оксана"},{"id":64,"external

[{'success': True,
  'uploadedOrders': [{'id': 45, 'externalId': 'id_77001234501_айгуль'},
   {'id': 46, 'externalId': 'id_77012345602_дина'},
   {'id': 47, 'externalId': 'id_77023456703_нургуль'},
   {'id': 48, 'externalId': 'id_77034567804_малика'},
   {'id': 49, 'externalId': 'id_77045678905_зарина'},
   {'id': 50, 'externalId': 'id_77056789006_ирина'},
   {'id': 51, 'externalId': 'id_77067890107_светлана'},
   {'id': 52, 'externalId': 'id_77078901208_гульнара'},
   {'id': 53, 'externalId': 'id_77089012309_анара'},
   {'id': 54, 'externalId': 'id_77090123410_камила'},
   {'id': 55, 'externalId': 'id_77001234511_жанар'},
   {'id': 56, 'externalId': 'id_77012345612_самал'},
   {'id': 57, 'externalId': 'id_77023456713_назгуль'},
   {'id': 58, 'externalId': 'id_77034567814_татьяна'},
   {'id': 59, 'externalId': 'id_77045678915_алия'},
   {'id': 60, 'externalId': 'id_77056789016_венера'},
   {'id': 61, 'externalId': 'id_77067890117_меруерт'},
   {'id': 62, 'externalId': 'id_77078901218_б

# RetailCRM → Supabase

## Окружение

In [6]:
import requests
import json
import time
from supabase import create_client, Client


In [84]:
TG_TOKEN = os.getenv("TG_TOKEN")
TG_CHAT_ID = os.getenv("TG_CHAT_ID")
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_SECRET_KEY = os.getenv("SUPABASE_SECRET_KEY")

## Функция парсинга заказов

In [21]:
# --- НАСТРОЙКИ ---
if SUPABASE_URL is None or SUPABASE_SECRET_KEY is None:
    raise ValueError(
        "Переменные окружения SUPABASE_URL и SUPABASE_SECRET_KEY должны быть заданы"
    )
# Инициализация клиента Supabase
supabase: Client = create_client(supabase_url=SUPABASE_URL, supabase_key=SUPABASE_SECRET_KEY)

In [ ]:
def fetch_orders_from_crm(subdomain, api_key):
    """
    Выгружает все заказы из RetailCRM постранично.
    :param subdomain: имя поддомена (например, 'shop-123')
    :param api_key: API ключ для доступа к CRM
    :return: список всех заказов, выгруженных из CRM
    """
    base_url = f"https://{subdomain}.retailcrm.ru/api/v5/orders"
    all_orders = []
    page = 1
    limit = 100 # Максимум по документации
    
    logger.info(f"Начинаю выгрузку заказов из поддомена {subdomain}...")

    # Используем Session для переиспользования соединения
    with requests.Session() as session:
        while True:
            params = {
                'apiKey': api_key,
                'limit': limit,
                'page': page
            }
            
            try:
                # GET запрос с таймаутом 10 секунд
                response = session.get(base_url, params=params, timeout=10)
                
                # Если сервер вернул ошибку (например, 401 или 500)
                if response.status_code != 200:
                    logger.error(f"Ошибка сервера {response.status_code} на странице {page}: {response.text}")
                
                response.raise_for_status()
                data = response.json()

                if not data.get('success'):
                    logger.error(f"Ошибка API на странице {page}: {data.get('errors', 'Неизвестная ошибка')}")
                    break

                orders = data.get('orders', [])
                if not orders:
                    logger.info("Больше заказов не найдено.")
                    break

                all_orders.extend(orders)
                
                pagination = data.get('pagination', {})
                total_pages = pagination.get('totalPageCount', 1)
                
                logger.info(f"Загружена страница {page} из {total_pages}. (Заказов в пакете: {len(orders)})")

                # Проверка: достигли ли мы последней страницы
                if page >= total_pages:
                    break
                
                page += 1
                
                # Короткая пауза, чтобы не нагружать API слишком сильно
                time.sleep(0.1)

            except Timeout:
                logger.error(f"Превышено время ожидания на странице {page} (timeout=10s)")
                break
            except RequestException as e:
                logger.error(f"Сетевая ошибка на странице {page}: {e}")
                break
            except Exception as e:
                logger.error(f"Непредвиденная ошибка на странице {page}: {e}")
                break
    
    
    logger.info(f"Выгрузка завершена. Всего собрано заказов: {len(all_orders)}")
    return all_orders


## Функция отправки уведомления в Telegram

In [94]:
def send_telegram_alert(token, chat_id, message):
    """Отправляет сообщение в Telegram через Bot API
    Вход:
        token - строка, токен бота Telegram (например, '123456:ABC-DEF1234ghIkl-zyx57W2v1u123ew11')
        chat_id - строка или число, идентификатор чата (например, '-1001234567890' для каналов или '@username' для личных сообщений)
        message - строка, текст сообщения, который нужно отправить
    """
    if not token or token == "None":
        logger.error("Ошибка: Telegram Token не задан!")
        return
    url = f"https://api.telegram.org/bot{token}/sendMessage"
    payload = {"chat_id": chat_id, "text": message, "parse_mode": "HTML"}
    try:
        response = requests.post(url, json=payload, timeout=5)
        response.raise_for_status()
    except Exception as e:
        logger.error(f"Ошибка отправки в Telegram: {e}")

## Функция валидации и отправки в Supabase

In [101]:
# --- Функция проверки (для создания заказа в Supabase) ---
def validate_with_pydantic_for_supabase(data_list):
    valid_orders = []
    errors = []

    for index, item in enumerate(data_list):
        try:
            # Пытаемся создать модель (здесь происходит валидация)
            order = OrderRead(**item)

            # ПРОВЕРКА УСЛОВИЯ: Сумма > 50,000 тенге
            if order.totalSumm > 50000:
                
                # Проверяем, что заказ свежий (например, создан не более 1 часа назад)
                # Это предотвратит повторные уведомления по старым заказам
                now = datetime.now()
                # Учитываем, что в заказе время может быть в другом часовом поясе
                time_diff = now - order.createdAt.replace(tzinfo=None) 
                
                if time_diff.total_seconds() < 3600: # 1 час
                    logger.info(f"Найшелся новый крупный заказ №{order.number} на сумму {order.totalSumm:,.0f} ₸")
                    msg = (
                        f"💰 <b>Заказ больше 50 000 ₸!</b>\n"
                        f"Номер: №{order.number}\n"
                        f"Клиент: {order.firstName}\n"
                        f"Сумма: <b>{order.totalSumm:,.0f} ₸</b>\n"
                        f"Статус: {order.status}"
                    )
                    send_telegram_alert(TG_TOKEN, TG_CHAT_ID, msg)
                    logger.info(f"Отправлено уведомление по заказу №{order.number}")

            # Если успешно, сохраняем очищенный словарь
            valid_orders.append(order.to_supabase())
        except Exception as e:
            errors.append(f"Заказ индекс {index}: {str(e)}")
    if errors:
        logger.warning(f"❌ Найдено ошибок при подготовке данных для Supabase: {len(errors)}")
        for err in errors:
            logger.error(err)
    else:
        logger.info(f"✅ Все заказы ({len(valid_orders)}) валидны и готовы к отправке в Supabase!")
   
    return valid_orders, errors

In [14]:
def sync_crm_to_supabase(subdomain, crm_token, supabase_client):
    """
    Основная функция синхронизации
    :param subdomain: имя поддомена в RetailCRM
    :param crm_token: API ключ для доступа к CRM
    :param supabase_client: инициализированный клиент Supabase
    :return: словарь для отправки в Supabase или None в случае ошибок
    """
    # 1. Выгружаем "сырые" данные из CRM
    # (используем ту функцию fetch_orders_from_crm, что мы написали ранее)
    raw_orders = fetch_orders_from_crm(subdomain, crm_token)
    
    if not raw_orders:
        logger.warning("Нет данных для синхронизации.")
        return

    # 2. Прогоняем через Pydantic для валидации и очистки
    logger.info("Начинаю валидацию данных через Pydantic...")
    
    validated_rows, _ = validate_with_pydantic_for_supabase(raw_orders)

    # 3. Загружаем в Supabase пакетами (Upsert)
    if validated_rows:
        logger.info(f"Загружаю {len(validated_rows)} заказов в Supabase...")
        try:
            # upsert обновит существующие ID и добавит новые
            # on_conflict='id' говорит, что 'id' - это уникальный ключ
            result = supabase_client.table("orders_crm").upsert(
                validated_rows, 
                on_conflict="id"
            ).execute()
            
            logger.info("✅ Синхронизация с Supabase завершена успешно!")
            return result
        except Exception as e:
            logger.error(f"Ошибка при вставке в Supabase: {e}")


## Выгрузка в Supabase (SQL)

In [25]:
# --- ЗАПУСК ---
sync_crm_to_supabase(SUBDOMAIN, API_KEY, supabase)

2026-04-13 21:28:20,110 - INFO - Начинаю выгрузку заказов из поддомена noscoks...
2026-04-13 21:28:21,907 - INFO - Загружена страница 1 из 1. (Заказов в пакете: 50)
2026-04-13 21:28:21,908 - INFO - Выгрузка завершена. Всего собрано заказов: 50
2026-04-13 21:28:21,912 - INFO - Начинаю валидацию данных через Pydantic...
2026-04-13 21:28:21,914 - INFO - ✅ Все заказы (50) валидны и готовы к отправке в Supabase!
2026-04-13 21:28:21,916 - INFO - Загружаю 50 заказов в Supabase...
2026-04-13 21:28:22,796 - INFO - HTTP Request: POST https://yuyfugkdieasojbxxbtm.supabase.co/rest/v1/orders_crm?on_conflict=id&columns=%22phone%22%2C%22number%22%2C%22created_at%22%2C%22external_id%22%2C%22last_name%22%2C%22total_sum%22%2C%22raw_data%22%2C%22first_name%22%2C%22status%22%2C%22id%22 "HTTP/2 200 OK"
2026-04-13 21:28:22,802 - INFO - ✅ Синхронизация с Supabase завершена успешно!


APIResponse(data=[{'id': 94, 'external_id': 'id_77090123450_феруза', 'number': '94A', 'first_name': 'Феруза', 'last_name': 'Юсупова', 'phone': '+77090123450', 'total_sum': 81000, 'status': 'offer-analog', 'created_at': '2026-04-13T00:56:01+00:00', 'raw_data': {'id': 94, 'email': 'feruza.yusupova@example.com', 'phone': '+77090123450', 'number': '94A', 'status': 'offer-analog', 'lastName': 'Юсупова', 'createdAt': '2026-04-13T00:56:01', 'firstName': 'Феруза', 'totalSumm': 81000.0, 'externalId': 'id_77090123450_феруза'}}, {'id': 93, 'external_id': 'id_77089012349_ажар', 'number': '93A', 'first_name': 'Ажар', 'last_name': 'Абенова', 'phone': '+77089012349', 'total_sum': 37000, 'status': 'offer-analog', 'created_at': '2026-04-13T00:56:01+00:00', 'raw_data': {'id': 93, 'email': 'azhar.abenova@example.com', 'phone': '+77089012349', 'number': '93A', 'status': 'offer-analog', 'lastName': 'Абенова', 'createdAt': '2026-04-13T00:56:01', 'firstName': 'Ажар', 'totalSumm': 37000.0, 'externalId': 'id_7

# Проверка уведомления в Telegram

In [102]:
# Закинем в CRM заказ на сумму больше 50,000 тенге
test_order = [
    {
        "firstName": "Новый Тестер ",
        "lastName": "Другой",
        "phone": "+70000000000",
        "email": "tester.drug@example.com",
        "orderType": "eshop-individual",
        "orderMethod": "shopping-cart",
        "status": "new",
        "items": [
            {
                "productName": "Утягивающее боди Nova Body",
                "quantity": 1,
                "initialPrice": 50005,
            }
        ],
        "delivery": {"address": {"city": "Алматы", "text": "ул. Ходжанова 76, кв 62"}},
        "customFields": {"utm_source": "instagram"},
    }
]
# Валидируем тестовый заказ перед загрузкой в CRM, чтобы убедиться, что он соответствует требованиям модели
clean_orders, validation_errors = validate_with_pydantic(test_order)

if validation_errors:
    print(f"❌ Найдено ошибок: {len(validation_errors)}")
    for err in validation_errors:
        print(err)
else:
    print(f"✅ Все заказы ({len(clean_orders)}) валидны и готовы к отправке!")
# Загружаем тестовый заказ в CRM
upload_orders_to_retailcrm(SUBDOMAIN, API_KEY, clean_orders, SITE)
# Теперь запускаем синхронизацию, чтобы увидеть, что уведомление в Telegram отправится по этому заказу
sync_crm_to_supabase(SUBDOMAIN, API_KEY, supabase)

✅ Все заказы (1) валидны и готовы к отправке!


2026-04-13 22:30:44,151 - ERROR - Пакет 1: ОШИБКА API: Order with externalId=id_70000000000_новый тестер  already exists.
2026-04-13 22:30:44,152 - ERROR - Пакет 1: Ошибка сети: 460 Client Error: unknown status for url: https://noscoks.retailcrm.ru/api/v5/orders/upload
2026-04-13 22:30:44,355 - INFO - Начинаю выгрузку заказов из поддомена noscoks...
2026-04-13 22:30:47,340 - INFO - Загружена страница 1 из 1. (Заказов в пакете: 52)
2026-04-13 22:30:47,341 - INFO - Выгрузка завершена. Всего собрано заказов: 52
2026-04-13 22:30:47,343 - INFO - Начинаю валидацию данных через Pydantic...
2026-04-13 22:30:47,346 - INFO - Найшелся новый крупный заказ №96A на сумму 50,005 ₸
2026-04-13 22:30:47,975 - INFO - Отправлено уведомление по заказу №96A
2026-04-13 22:30:47,976 - INFO - Найшелся новый крупный заказ №95A на сумму 150,003 ₸
2026-04-13 22:30:48,647 - INFO - Отправлено уведомление по заказу №95A
2026-04-13 22:30:48,649 - INFO - ✅ Все заказы (52) валидны и готовы к отправке в Supabase!
2026-0

APIResponse(data=[{'id': 96, 'external_id': 'id_70000000000_новый тестер ', 'number': '96A', 'first_name': 'Новый Тестер ', 'last_name': 'Другой', 'phone': '+70000000000', 'total_sum': 50005, 'status': 'offer-analog', 'created_at': '2026-04-13T22:28:57+00:00', 'raw_data': {'id': 96, 'email': 'tester.drug@example.com', 'phone': '+70000000000', 'number': '96A', 'status': 'offer-analog', 'lastName': 'Другой', 'createdAt': '2026-04-13T22:28:57', 'firstName': 'Новый Тестер ', 'totalSumm': 50005.0, 'externalId': 'id_70000000000_новый тестер '}}, {'id': 95, 'external_id': 'id_70000000000_тестер', 'number': '95A', 'first_name': 'Тестер', 'last_name': 'Супабейсов', 'phone': '+70000000000', 'total_sum': 150003, 'status': 'offer-analog', 'created_at': '2026-04-13T21:39:38+00:00', 'raw_data': {'id': 95, 'email': 'tester.supabeysov@example.com', 'phone': '+70000000000', 'number': '95A', 'status': 'offer-analog', 'lastName': 'Супабейсов', 'createdAt': '2026-04-13T21:39:38', 'firstName': 'Тестер', 't